In [1]:
# Save this as test_api.py and run it
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[{
        "role": "user",
        "content": "Reply with just: API working"
    }]
)

print(message.content[0].text)
# Should print: API working

API working


In [2]:
from db.execute import run_sql
df = run_sql('SELECT * FROM samples.tpch.orders LIMIT 5')
print(df.to_string())

   o_orderkey  o_custkey o_orderstatus o_totalprice o_orderdate  o_orderpriority          o_clerk  o_shippriority                                                           o_comment
0    11396166     179329             O    187683.10  1996-06-22  4-NOT SPECIFIED  Clerk#000004689               0              ep fluffily regular packages. regular, final courts ag
1    11396167     473245             F    117554.58  1994-07-20         1-URGENT  Clerk#000001856               0                efully after the carefully express packages; final r
2    11396192      77549             O     47611.56  1996-01-03         3-MEDIUM  Clerk#000001450               0                                    es. regular tithes poach careful
3    11396193     610651             F    121869.26  1994-11-29            5-LOW  Clerk#000003696               0  are furiously along the bold, even ideas. even, final pinto beans 
4    11396194     193169             O    241066.47  1998-03-21  4-NOT SPECIFIED  Clerk#00

In [3]:
import chromadb
client = chromadb.PersistentClient(path='data/chroma_db')
col = client.get_collection('knowledge_base')
results = col.get(include=['documents', 'metadatas'])
for i, (doc, meta) in enumerate(zip(results['documents'], results['metadatas'])):
    print(f'--- Chunk {i+1} | Source: {meta} ---')
    print(doc[:300])
    print()

--- Chunk 1 | Source: {'source': 'data_dictionary.pdf', 'chunk_index': 0} ---
Data Dictionary
TPC-H Schema Reference — Databricks samples.tpch
This document is the authoritative data dictionary for the TPC-H analytics database available in Databricks
as samples.tpch. It describes every table, its business purpose, all columns with data types and
descriptions, primary and fore

--- Chunk 2 | Source: {'chunk_index': 1, 'source': 'data_dictionary.pdf'} ---
 this document to understand
what data exists before writing queries.
Catalog: samples    Schema: tpch    Total tables: 8    Approx total rows: ~7.5 million
1. ORDERS
Full name: samples.tpch.orders
Approximate rows: ~1,500,000    Primary key: o_orderkey
Foreign keys: o_custkey → customer.c_custkey
T

--- Chunk 3 | Source: {'chunk_index': 2, 'source': 'data_dictionary.pdf'} ---
er placed on the platform. Each row represents one
order. This is the starting point for most revenue, volume, and customer analysis queries.
Column
Type
Descript

In [4]:
import chromadb
client = chromadb.PersistentClient(path='data/chroma_db')
col = client.get_collection('knowledge_base')
results = col.query(query_texts=['What is Average Order Value'], n_results=3)
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f'--- Result {i+1} | {meta} ---')
    print(doc)
    print()

--- Result 1 | {'source': 'metrics_definitions.pdf', 'chunk_index': 4} ---
imary indicator of sales quality.
Example query: What is the Average Order Value per market segment?
1.3 Revenue by Region / Nation

Definition: Total revenue broken down by geographic region or nation, used for regional performance
comparison and territory management.
Tables used: lineitem, orders, customer, nation, region
Join path: lineitem → orders (o_orderkey) → customer (o_custkey) → nation 

--- Result 2 | {'chunk_index': 6, 'source': 'metrics_definitions.pdf'} ---
ve been shipped, indicating operational
efficiency of the supply chain.
Formula: COUNT(orders where all items shipped) / COUNT(total orders) * 100
Tables used: samples.tpch.orders, samples.tpch.lineitem
Order status values: O = Open, F = Fulfilled, P = Partially shipped
2.3 Average Days to Ship
Definition: The average number of days between an order being placed and the line items being shipped. A

--- Result 3 | {'source': 'metrics_definitions

In [5]:
import sys, os
sys.path.insert(0, "/Users/piyushraj/code/AI08/Capstone1_Implementation")
os.chdir("/Users/piyushraj/code/AI08/Capstone1_Implementation")

In [6]:
from agents.orchestrator import agent_graph

In [7]:
state = agent_graph.invoke({
    "question": "From when to when are the orders stored in the database?",
    "user_role": "analyst"
})
print("Intent   :", state.get("intent"))
print("Answer   :", state.get("final_answer"))
print("Error    :", state.get("error"))

Intent   : sql_query
Answer   : The orders in the database span from **January 1, 1992 to August 2, 1998**, covering approximately 6.5 years of historical order data. This represents the complete date range of stored orders in the system.
Error    : 


In [8]:
state = agent_graph.invoke({
    "question": "What is Average Order Value?",
    "user_role": "analyst"
})
print("Intent   :", state.get("intent"))
print("Answer   :", state.get("final_answer"))
print("Error    :", state.get("error"))

Intent   : doc_lookup
Answer   : Based on the context provided:

**Average Order Value (AOV)** is calculated using the following formula:

**SUM(l_extendedprice * (1 - l_discount)) / COUNT(DISTINCT o_orderkey)**

**Definition:** The total revenue from a single customer order, calculated by dividing total revenue by the number of distinct orders.

**Tables used:** samples.tpch.lineitem, samples.tpch.orders

**Interpretation:** A rising AOV indicates customers are purchasing higher-value items or placing larger orders. It is a primary indicator of sales quality.
Error    : None


In [8]:
from db.execute import run_sql
df = run_sql("SELECT MIN(o_orderdate) as min_date, MAX(o_orderdate) as max_date FROM samples.tpch.orders")
display(df)

,min_date,max_date
0,1992-01-01,1998-08-02


In [9]:
state = agent_graph.invoke({
    "question": "Show total revenue per nation for year 1997, joining lineitem, orders, customer and nation tables, ordered highest to lowest",
    "user_role": "finance"
})
print("Intent   :", state.get("intent"))
print("SQL      :", state.get("generated_sql"))
print("Error    :", state.get("error"))
if state.get("result_df") is not None:
    display(state["result_df"])

Intent   : sql_query
SQL      : SELECT 
  n.n_name,
  SUM(l.l_extendedprice * (1 - l.l_discount)) as total_revenue
FROM samples.tpch.lineitem l
JOIN samples.tpch.orders o ON l.l_orderkey = o.o_orderkey
JOIN samples.tpch.supplier s ON l.l_suppkey = s.s_suppkey
JOIN samples.tpch.nation n ON s.s_nationkey = n.n_nationkey
WHERE YEAR(o.o_orderdate) = 1997
GROUP BY n.n_name
ORDER BY total_revenue DESC
Error    : 


,n_name,total_revenue
0,INDIA,6837033129.3472
1,ARGENTINA,6797203920.3324
2,INDONESIA,6765958583.5012
3,IRAQ,6757891655.0479
4,UNITED KINGDOM,6735905803.5481
5,CANADA,6714989634.8571
6,SAUDI ARABIA,6680268809.6875
7,MOROCCO,6634532413.4675
8,IRAN,6633749462.0413
9,EGYPT,6623362709.0112


In [10]:
state = agent_graph.invoke({
    "question": "Show me the top 10 customer name by total order value in 1997",
    "user_role": "finance"
})
print("Role     :", state.get("user_role"))
print("Intent   :", state.get("intent"))
print("SQL      :", state.get("generated_sql"))
print("Error    :", state.get("error"))
if state.get("result_df") is not None:
    display(state["result_df"])

Role     : finance
Intent   : sql_query
SQL      : 
Error    : RBAC violation: role 'finance' is not allowed to access: ['samples.tpch.customer', 'samples.tpch.part']


In [11]:
state = agent_graph.invoke({
    "question": "Show me the top 10 customer name by total order value in 1997",
    "user_role": "analyst"
})
print("Role     :", state.get("user_role"))
print("Intent   :", state.get("intent"))
print("SQL      :", state.get("generated_sql"))
print("Error    :", state.get("error"))
if state.get("result_df") is not None:
    display(state["result_df"])


Role     : analyst
Intent   : sql_query
SQL      : SELECT 
  c.c_name,
  SUM(o.o_totalprice) AS total_order_value
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
WHERE YEAR(o.o_orderdate) = 1997
GROUP BY c.c_name
ORDER BY total_order_value DESC
LIMIT 10
Error    : 


,c_name,total_order_value
0,Customer#000264430,2470586.66
1,Customer#000258772,2389902.88
2,Customer#000308521,2312814.88
3,Customer#000036460,2275269.15
4,Customer#000691849,2246039.57
5,Customer#000414049,2236777.09
6,Customer#000424813,2225530.55
7,Customer#000210148,2224308.16
8,Customer#000117304,2216460.36
9,Customer#000012874,2204260.53


In [12]:
state = agent_graph.invoke({
    "question": "When was the orders table last updated?",
    "user_role": "executive"
})
print("Intent   :", state.get("intent"))
print("Answer   :", state.get("final_answer"))
print("Error    :", state.get("error"))

Intent   : metadata
Answer   : Table `samples.tpch.orders` was last updated on **2026-04-23 15:09:12+00:00** via operation: `CREATE OR REPLACE TABLE AS SELECT`.
Error    : None
